In [ ]:
# If using Google Colab

from google.colab import userdata
API_KEY = userdata.get('APCA_API_KEY_ID')
API_SECRET = userdata.get('APCA_API_SECRET_KEY')

# If on your own machine or VPS
# import os
# API_KEY = os.getenv("APCA_API_KEY_ID")
# API_SECRET = os.getenv("APCA_API_SECRET_KEY")

In [ ]:
# alpaca-py is not automatically installed on Colab
!pip install alpaca-py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.5/122.5 kB 3.9 MB/s eta 0:00:00


In [ ]:
from alpaca.trading.client import TradingClient

trading_client = TradingClient(
   API_KEY,
   API_SECRET,
   paper=True
)

In [ ]:
from alpaca.data.historical.stock import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

from datetime import datetime, timedelta

data_client = StockHistoricalDataClient(
    API_KEY,
    API_SECRET
)

def get_prices(symbol="SPY"):
    request = StockBarsRequest(
        symbol_or_symbols=[symbol],
        timeframe=TimeFrame.Day,
        start=datetime.now() - timedelta(days=50),
    )

    bars = data_client.get_stock_bars(request)

    return bars.df["close"]

In [ ]:
def generate_signal(prices):
    short_ma = prices.rolling(10).mean().iloc[-1]
    long_ma = prices.rolling(30).mean().iloc[-1]

    print("short_ma:", short_ma)
    print("long_ma:", long_ma)

    if short_ma > long_ma:
        return 1

    return 0

In [ ]:
def get_current_position(symbol):
    try:
        position = trading_client.get_open_position(symbol)
        return int(position.qty)
    except:
        return 0

In [ ]:
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide
from alpaca.trading.enums import TimeInForce

def submit_order(symbol, side, quantity):

    order = MarketOrderRequest(
        symbol=symbol,
        qty=quantity,
        side=side,
        time_in_force=TimeInForce.DAY
    )

    trading_client.submit_order(order)

In [ ]:
def run_strategy():

    symbol = "SPY"

    prices = get_prices(symbol)
    print("len(prices):", len(prices))

    target_position = generate_signal(prices)
    print("target_position:", target_position)

    current_shares = get_current_position(symbol)
    print("current_shares:", current_shares)

    if target_position == 1 and current_shares == 0:

        submit_order(
            symbol,
            OrderSide.BUY,
            1
        )

        print("Entered long position")

    elif target_position == 0 and current_shares > 0:

        submit_order(
            symbol,
            OrderSide.SELL,
            current_shares
        )

        print("Exited position")

    else:
        print("No action taken")

In [ ]:
run_strategy()

len(prices): 35
short_ma: 741.162
long_ma: 744.3836666666667
target_position: 0
current_shares: 0
No action taken
